# Phase 1: Environment Setup & Raw Schema Inspection

### Objective
Initialize the processing environment and set up memory-efficient data scanning for the LendingClub dataset using Polars LazyFrames.

### Technical Rationale
The raw LendingClub CSV is multi-gigabyte containing over 2 million rows and 100+ feature columns. Standard Pandas `read_csv` attempts to load the entire payload into RAM at once, leading to excessive memory pressure or out-of-memory (OOM) crashes.
- We use `pl.scan_csv` to establish a query plan to explore the dataset

In [1]:
import polars as pl

# Polars display config
pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(12)

# File path 
DATA_PATH = "/home/mike/Documents/work/DS/projects/credit-risk/data/raw/accepted_2007_to_2018Q4.csv"

# Step 1: Create the LazyFrame query graph
lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True
)

In [4]:
lazy_df

In [6]:
schema = lazy_df.collect_schema()
total_columns = len(schema.names())

print(f"Total Columns Loaded: {total_columns}")

Total Columns Loaded: 151


In [7]:
lazy_df.head(5).collect()

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,…,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
i64,str,f64,f64,f64,str,…,str,str,str,f64,f64,f64
68407277,null,3600.0,3600.0,3600.0,""" 36 months""",…,null,null,null,null,null,null
68355089,null,24700.0,24700.0,24700.0,""" 36 months""",…,null,null,null,null,null,null
68341763,null,20000.0,20000.0,20000.0,""" 60 months""",…,null,null,null,null,null,null
66310712,null,35000.0,35000.0,35000.0,""" 60 months""",…,null,null,null,null,null,null
68476807,null,10400.0,10400.0,10400.0,""" 60 months""",…,null,null,null,null,null,null


In [5]:
#rows with text in the id column
bad_rows = df_lazy.filter(pl.col("id").str.contains("Total")).collect()
print(bad_rows)

shape: (32, 151)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ member_id ┆ loan_amnt ┆ funded_am ┆ … ┆ settlemen ┆ settlemen ┆ settlemen ┆ settleme │
│ ---       ┆ ---       ┆ ---       ┆ nt        ┆   ┆ t_date    ┆ t_amount  ┆ t_percent ┆ nt_term  │
│ str       ┆ str       ┆ f64       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ age       ┆ ---      │
│           ┆           ┆           ┆ f64       ┆   ┆ str       ┆ f64       ┆ ---       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Total     ┆ null      ┆ null      ┆ null      ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│ amount    ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ funded in ┆           ┆           ┆           ┆   ┆           ┆         

In [6]:
#Remove the bad rows
credit_risk = df_lazy.filter(~pl.col("id").str.contains("Total")).collect()
credit_risk.head(5)

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,…,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,str,str,str,f64,f64,f64
"""68407277""",null,3600.0,3600.0,3600.0,""" 36 months""",13.99,123.03,"""C""","""C4""","""leadman""","""10+ years""","""MORTGAGE""",55000.0,"""Not Verified""","""Dec-2015""","""Fully Paid""","""n""","""https://lendingclub.com/browse…",null,"""debt_consolidation""","""Debt consolidation""","""190xx""","""PA""",5.91,0.0,"""Aug-2003""",675.0,679.0,1.0,30.0,null,7.0,0.0,2765.0,29.7,13.0,…,13734.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Cash""","""N""",null,null,null,null,null,null
"""68355089""",null,24700.0,24700.0,24700.0,""" 36 months""",11.99,820.28,"""C""","""C1""","""Engineer""","""10+ years""","""MORTGAGE""",65000.0,"""Not Verified""","""Dec-2015""","""Fully Paid""","""n""","""https://lendingclub.com/browse…",null,"""small_business""","""Business""","""577xx""","""SD""",16.06,1.0,"""Dec-1999""",715.0,719.0,4.0,6.0,null,22.0,0.0,21470.0,19.2,38.0,…,24667.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Cash""","""N""",null,null,null,null,null,null
"""68341763""",null,20000.0,20000.0,20000.0,""" 60 months""",10.78,432.66,"""B""","""B4""","""truck driver""","""10+ years""","""MORTGAGE""",63000.0,"""Not Verified""","""Dec-2015""","""Fully Paid""","""n""","""https://lendingclub.com/browse…",null,"""home_improvement""",null,"""605xx""","""IL""",10.78,0.0,"""Aug-2000""",695.0,699.0,0.0,null,null,6.0,0.0,7869.0,56.2,18.0,…,14877.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Cash""","""N""",null,null,null,null,null,null
"""66310712""",null,35000.0,35000.0,35000.0,""" 60 months""",14.85,829.9,"""C""","""C5""","""Information Systems Officer""","""10+ years""","""MORTGAGE""",110000.0,"""Source Verified""","""Dec-2015""","""Current""","""n""","""https://lendingclub.com/browse…",null,"""debt_consolidation""","""Debt consolidation""","""076xx""","""NJ""",17.06,0.0,"""Sep-2008""",785.0,789.0,0.0,null,null,13.0,0.0,7802.0,11.6,17.0,…,18000.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Cash""","""N""",null,null,null,null,null,null
"""68476807""",null,10400.0,10400.0,10400.0,""" 60 months""",22.45,289.91,"""F""","""F1""","""Contract Specialist""","""3 years""","""

In [7]:
type(credit_risk)

polars.dataframe.frame.DataFrame